In [1]:
from skillcorner.client import SkillcornerClient
from pprint import pprint
import pandas as pd
from io import BytesIO
from IPython.display import display

In [2]:
import os
USERNAME = os.environ["SC_USERNAME"]
PASSWORD = os.environ["SC_PASSWORD"]
DATA_DIR = os.environ.get("DATA_DIR", os.path.expanduser("~/Desktop/analisis_ca/scout_dashboard/data"))

client = SkillcornerClient(username=USERNAME, password=PASSWORD)
client

In [ ]:
# ID DE LIGAS

# Liga MX: 97 y 610
# Concachampions: 159
# Argentina: 70
# Brasil: 77
# Colombia: 100
# Ecuador: 163
# Paraguay: 140
# Uruguay: 95
# MLS: 100
# La Liga: 4
# La Liga 2: 42
# Chile: 57
# Premier League: 1
# Championship: 31
# Serie A: 5
# Serie B: 41
# Bundesliga: 6
# Bundesliga 2: 40
# Ligue 1: 3
# Eredivisie: 17
# Bélgica: 16
# Portugal: 25
# Turquía: 29
# Escocia: 18
# UCL: 10
# UEL: 9
# Libertadores: 109

In [4]:
# ID DE TEMPORADAS

# 25/26: 129
# 2025: 128
# 2026: 130

In [ ]:
# LIGA MX

# PHYSICAL DATA STANDARDIZED AS CSV - COMBINED COMPETITIONS, BOTH SEASONS
#
# Pulls 25/26 (season 129) and 26/27 (season 131) separately -- each season
# aggregates its own comp_id 610+97 split the same way as before (two
# editions of the same season, legitimately mergeable), then the two
# seasons' results are appended as separate rows, never blended together.
# Averaging/maxing a fully-played season together with a brand-new one
# that has only a few matches so far would produce a number that doesn't
# correspond to either real season, and would collapse the dashboard's
# own season_name selector (and "Acumulado") into a single row -- the app
# already knows how to combine seasons per player correctly; this file
# just needs to carry both so it can.

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

for season_id in [129, 131]:  # 25/26, 26/27
    dfs = []
    for comp_id in [610, 97]:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }

        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    # Metadata columns — don't aggregate these
    # season_id/competition_id (and their _name siblings) excluded too --
    # they're numeric, so without this they'd get summed across the two
    # comp_ids just like a real metric would (season_id 131+131=262),
    # corrupting the season join key downstream (see arg/col cells for
    # where this exact bug was first caught).
    meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                 "season_id", "season_name", "competition_id", "competition_name"]

    # Numeric columns split into two aggregation strategies when a player
    # appears in both competition pulls (610 and 97) *within this season*:
    #  - rate/peak-style metrics (per-60-min rates, per-minute rates, peak
    #    speed, percentages/ratios) are NOT additive across two halves of a
    #    season -- summing them silently doubles values like PSV99 (peak
    #    sprint velocity), which showed up as ~58 km/h in the dashboard, about
    #    2x a realistic human top speed. These use max instead.
    #  - genuinely cumulative counters (total minutes played, match counts,
    #    raw distance/count totals not already normalized to a rate) are
    #    correctly additive across the two halves and keep using sum.
    rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
    numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ['float64', 'int64']]
    max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
    sum_cols = [c for c in numeric_cols if c not in max_cols]

    print(f"Season {season_id} — Aggregating as MAX (rate/peak metrics):", max_cols)
    print(f"Season {season_id} — Aggregating as SUM (cumulative totals):", sum_cols)

    # Preserve all non-numeric, non-groupby columns (team_id, player_short_name,
    # player_birthdate, season_name, season_id, competition_id, etc.) as "last"
    other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]
    df_agg = df.groupby(["player_id", "player_name"], as_index=False).agg(
        **{col: (col, "last") for col in other_meta},
        **{col: (col, "sum") for col in sum_cols},
        **{col: (col, "max") for col in max_cols}
    )

    # Re-run normalization on aggregated totals
    before = set(df_agg.columns)
    p_utils.add_standard_metrics(df_agg)
    added = sorted(set(df_agg.columns) - before)

    print(f"Season {season_id} — Added columns:", added)
    print(f"Season {season_id} — Has minutes_full_tip?", "minutes_full_tip" in df_agg.columns)

    season_dfs.append(df_agg)

df_agg = pd.concat(season_dfs, ignore_index=True)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "ligamx_physical_standardized.csv")
df_agg.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df_agg.shape)

# from skillcorner.client import SkillcornerClient
# from skillcornerviz.utils import skillcorner_physical_utils as p_utils
# import pandas as pd
# import os

# client = SkillcornerClient(username=USERNAME, password=PASSWORD)

# params = {
#     "season": 129,
#     "competition": 97,
#     "group_by": "player,team,competition,season,group",

#     # required so minutes_full_tip exists
#     "possession": "all,tip,otip",

#     # often required to get the v3 schema fields used by p_utils
#     "data_version": "3",

#     "playing_time__gte": 45,
#     "count_match__gte": 5,
# }

# data = client.get_physical(params=params)
# df = pd.DataFrame(data)

# before = set(df.columns)
# p_utils.add_standard_metrics(df)     # mutates df (adds columns)
# added = sorted(set(df.columns) - before)

# print("Added columns:")
# print(added)

# print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

# desktop_path = os.path.join(DATA_DIR, "physical")
# os.makedirs(desktop_path, exist_ok=True)

# file_path = os.path.join(desktop_path, "ligamx_physical_standardized.csv")
# df.to_csv(file_path, index=False)

# print("Saved to:", file_path)
# print("Shape:", df.shape)

In [ ]:
# ARGENTINA
#
# 2025 (season 128): single comp 70. 2026 (season 130): split into 1st Phase (70) + 2nd Phase (376, a new id that only exists for 2026) -- both seasons kept as separate rows, never blended (see Liga MX cell for why).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_plan = [
    (128, [70]),  # 2025
    (130, [70, 376]),  # 2026
]

season_dfs = []

for season_id, comp_ids in season_plan:
    dfs = []
    for comp_id in comp_ids:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }
        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    if len(comp_ids) > 1:
        # Numeric columns split into two aggregation strategies when a player
        # appears in both competition pulls *within this season* (e.g. two
        # Apertura/Clausura editions):
        #  - rate/peak-style metrics (per-60-min rates, per-minute rates, peak
        #    speed, percentages/ratios) are NOT additive across two halves of a
        #    season -- summing them silently doubles values like PSV99 (peak
        #    sprint velocity). These use max instead.
        #  - genuinely cumulative counters (minutes played, match counts, raw
        #    distance/count totals) are correctly additive and keep using sum.
        #  See notebooks/physical.ipynb's Liga MX cell for where this pattern
        #  and the PSV99 bug it fixes were first worked out.
        # season_id/competition_id (and their _name siblings) must stay out of
        # sum_cols too -- they're numeric, so without this they'd silently get
        # summed across the two comp_ids just like a real metric would (e.g.
        # season_id 130+130=260), corrupting the season join key downstream.
        meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                     "season_id", "season_name", "competition_id", "competition_name"]
        rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
        numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
        max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
        sum_cols = [c for c in numeric_cols if c not in max_cols]
        other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

        df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
            **{col: (col, "last") for col in other_meta},
            **{col: (col, "sum") for col in sum_cols},
            **{col: (col, "max") for col in max_cols}
        )
    else:
        # Single competition edition for this season -- nothing to combine,
        # keep every column (including season_id/competition_id) untouched.
        df_season = df

    before = set(df_season.columns)
    p_utils.add_standard_metrics(df_season)
    added = sorted(set(df_season.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df_season.columns)

    season_dfs.append(df_season)

df = pd.concat(season_dfs, ignore_index=True)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "arg_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)


In [ ]:
# BRASIL
#
# Série A, comp 77 stable across years -- add the 2026 season (130) alongside 2025 (128), kept as separate rows.

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_plan = [
    (128, [77]),  # 2025
    (130, [77]),  # 2026
]

season_dfs = []

for season_id, comp_ids in season_plan:
    dfs = []
    for comp_id in comp_ids:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }
        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    if len(comp_ids) > 1:
        # Numeric columns split into two aggregation strategies when a player
        # appears in both competition pulls *within this season* (e.g. two
        # Apertura/Clausura editions):
        #  - rate/peak-style metrics (per-60-min rates, per-minute rates, peak
        #    speed, percentages/ratios) are NOT additive across two halves of a
        #    season -- summing them silently doubles values like PSV99 (peak
        #    sprint velocity). These use max instead.
        #  - genuinely cumulative counters (minutes played, match counts, raw
        #    distance/count totals) are correctly additive and keep using sum.
        #  See notebooks/physical.ipynb's Liga MX cell for where this pattern
        #  and the PSV99 bug it fixes were first worked out.
        # season_id/competition_id (and their _name siblings) must stay out of
        # sum_cols too -- they're numeric, so without this they'd silently get
        # summed across the two comp_ids just like a real metric would (e.g.
        # season_id 130+130=260), corrupting the season join key downstream.
        meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                     "season_id", "season_name", "competition_id", "competition_name"]
        rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
        numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
        max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
        sum_cols = [c for c in numeric_cols if c not in max_cols]
        other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

        df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
            **{col: (col, "last") for col in other_meta},
            **{col: (col, "sum") for col in sum_cols},
            **{col: (col, "max") for col in max_cols}
        )
    else:
        # Single competition edition for this season -- nothing to combine,
        # keep every column (including season_id/competition_id) untouched.
        df_season = df

    before = set(df_season.columns)
    p_utils.add_standard_metrics(df_season)
    added = sorted(set(df_season.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df_season.columns)

    season_dfs.append(df_season)

df = pd.concat(season_dfs, ignore_index=True)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "brasil_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)


In [ ]:
# COLOMBIA
#
# 2025 (season 128): single comp 100 (Liga Dimayor). 2026 (season 130): split into Liga Dimayor (100) + Liga Dimayor Clausura (653, a new id that only exists for 2026) -- both seasons kept as separate rows.

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_plan = [
    (128, [100]),  # 2025
    (130, [100, 653]),  # 2026
]

season_dfs = []

for season_id, comp_ids in season_plan:
    dfs = []
    for comp_id in comp_ids:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }
        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    if len(comp_ids) > 1:
        # Numeric columns split into two aggregation strategies when a player
        # appears in both competition pulls *within this season* (e.g. two
        # Apertura/Clausura editions):
        #  - rate/peak-style metrics (per-60-min rates, per-minute rates, peak
        #    speed, percentages/ratios) are NOT additive across two halves of a
        #    season -- summing them silently doubles values like PSV99 (peak
        #    sprint velocity). These use max instead.
        #  - genuinely cumulative counters (minutes played, match counts, raw
        #    distance/count totals) are correctly additive and keep using sum.
        #  See notebooks/physical.ipynb's Liga MX cell for where this pattern
        #  and the PSV99 bug it fixes were first worked out.
        # season_id/competition_id (and their _name siblings) must stay out of
        # sum_cols too -- they're numeric, so without this they'd silently get
        # summed across the two comp_ids just like a real metric would (e.g.
        # season_id 130+130=260), corrupting the season join key downstream.
        meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                     "season_id", "season_name", "competition_id", "competition_name"]
        rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
        numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
        max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
        sum_cols = [c for c in numeric_cols if c not in max_cols]
        other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

        df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
            **{col: (col, "last") for col in other_meta},
            **{col: (col, "sum") for col in sum_cols},
            **{col: (col, "max") for col in max_cols}
        )
    else:
        # Single competition edition for this season -- nothing to combine,
        # keep every column (including season_id/competition_id) untouched.
        df_season = df

    before = set(df_season.columns)
    p_utils.add_standard_metrics(df_season)
    added = sorted(set(df_season.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df_season.columns)

    season_dfs.append(df_season)

df = pd.concat(season_dfs, ignore_index=True)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "col_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)


In [ ]:
# CHILE
#
# Primera Division, comp 57 stable across years -- add the 2026 season (130) alongside 2025 (128), kept as separate rows.

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_plan = [
    (128, [57]),  # 2025
    (130, [57]),  # 2026
]

season_dfs = []

for season_id, comp_ids in season_plan:
    dfs = []
    for comp_id in comp_ids:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }
        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    if len(comp_ids) > 1:
        # Numeric columns split into two aggregation strategies when a player
        # appears in both competition pulls *within this season* (e.g. two
        # Apertura/Clausura editions):
        #  - rate/peak-style metrics (per-60-min rates, per-minute rates, peak
        #    speed, percentages/ratios) are NOT additive across two halves of a
        #    season -- summing them silently doubles values like PSV99 (peak
        #    sprint velocity). These use max instead.
        #  - genuinely cumulative counters (minutes played, match counts, raw
        #    distance/count totals) are correctly additive and keep using sum.
        #  See notebooks/physical.ipynb's Liga MX cell for where this pattern
        #  and the PSV99 bug it fixes were first worked out.
        # season_id/competition_id (and their _name siblings) must stay out of
        # sum_cols too -- they're numeric, so without this they'd silently get
        # summed across the two comp_ids just like a real metric would (e.g.
        # season_id 130+130=260), corrupting the season join key downstream.
        meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                     "season_id", "season_name", "competition_id", "competition_name"]
        rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
        numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
        max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
        sum_cols = [c for c in numeric_cols if c not in max_cols]
        other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

        df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
            **{col: (col, "last") for col in other_meta},
            **{col: (col, "sum") for col in sum_cols},
            **{col: (col, "max") for col in max_cols}
        )
    else:
        # Single competition edition for this season -- nothing to combine,
        # keep every column (including season_id/competition_id) untouched.
        df_season = df

    before = set(df_season.columns)
    p_utils.add_standard_metrics(df_season)
    added = sorted(set(df_season.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df_season.columns)

    season_dfs.append(df_season)

df = pd.concat(season_dfs, ignore_index=True)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "chile_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)


In [ ]:
# MLS
#
# Major League Soccer, comp 60 stable across years -- add the 2026 season (130) alongside 2025 (128), kept as separate rows.

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_plan = [
    (128, [60]),  # 2025
    (130, [60]),  # 2026
]

season_dfs = []

for season_id, comp_ids in season_plan:
    dfs = []
    for comp_id in comp_ids:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }
        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    if len(comp_ids) > 1:
        # Numeric columns split into two aggregation strategies when a player
        # appears in both competition pulls *within this season* (e.g. two
        # Apertura/Clausura editions):
        #  - rate/peak-style metrics (per-60-min rates, per-minute rates, peak
        #    speed, percentages/ratios) are NOT additive across two halves of a
        #    season -- summing them silently doubles values like PSV99 (peak
        #    sprint velocity). These use max instead.
        #  - genuinely cumulative counters (minutes played, match counts, raw
        #    distance/count totals) are correctly additive and keep using sum.
        #  See notebooks/physical.ipynb's Liga MX cell for where this pattern
        #  and the PSV99 bug it fixes were first worked out.
        # season_id/competition_id (and their _name siblings) must stay out of
        # sum_cols too -- they're numeric, so without this they'd silently get
        # summed across the two comp_ids just like a real metric would (e.g.
        # season_id 130+130=260), corrupting the season join key downstream.
        meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                     "season_id", "season_name", "competition_id", "competition_name"]
        rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
        numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
        max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
        sum_cols = [c for c in numeric_cols if c not in max_cols]
        other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

        df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
            **{col: (col, "last") for col in other_meta},
            **{col: (col, "sum") for col in sum_cols},
            **{col: (col, "max") for col in max_cols}
        )
    else:
        # Single competition edition for this season -- nothing to combine,
        # keep every column (including season_id/competition_id) untouched.
        df_season = df

    before = set(df_season.columns)
    p_utils.add_standard_metrics(df_season)
    added = sorted(set(df_season.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df_season.columns)

    season_dfs.append(df_season)

df = pd.concat(season_dfs, ignore_index=True)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "mls_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)


In [12]:
# PREMIER LEAGUE


from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

params = {
    "season": 129,
    "competition": 1,
    "group_by": "player,team,competition,season,group",

    # required so minutes_full_tip exists
    "possession": "all,tip,otip",

    # often required to get the v3 schema fields used by p_utils
    "data_version": "3",

    "playing_time__gte": 45,
    "count_match__gte": 5,
}

data = client.get_physical(params=params)
df = pd.DataFrame(data)

before = set(df.columns)
p_utils.add_standard_metrics(df)     # mutates df (adds columns)
added = sorted(set(df.columns) - before)

print("Added columns:")
print(added)

print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "premier_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)

Added columns:
['accel_count_full_all', 'accel_count_full_bip', 'accel_count_full_otip', 'accel_count_full_tip', 'accel_count_per_30_otip', 'accel_count_per_30_tip', 'accel_count_per_60_bip', 'accel_count_per_90', 'decel_count_full_all', 'decel_count_full_bip', 'decel_count_full_otip', 'decel_count_full_tip', 'decel_count_per_30_otip', 'decel_count_per_30_tip', 'decel_count_per_60_bip', 'decel_count_per_90', 'distance_per_sprint', 'distance_per_sprint_bip', 'distance_per_sprint_otip', 'distance_per_sprint_tip', 'hi_count_full_bip', 'hi_count_per_30_otip', 'hi_count_per_30_tip', 'hi_count_per_60_bip', 'hi_count_per_90', 'hi_distance_full_bip', 'hi_distance_per_30_otip', 'hi_distance_per_30_tip', 'hi_distance_per_60_bip', 'hi_distance_per_90', 'hi_meters_per_minute', 'hi_meters_per_minute_bip', 'hi_meters_per_minute_otip', 'hi_meters_per_minute_tip', 'highaccel_count_full_bip', 'highaccel_count_per_30_otip', 'highaccel_count_per_30_tip', 'highaccel_count_per_60_bip', 'highaccel_count_per

In [13]:
# LA LIGA

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

params = {
    "season": 129,
    "competition": 4,
    "group_by": "player,team,competition,season,group",

    # required so minutes_full_tip exists
    "possession": "all,tip,otip",

    # often required to get the v3 schema fields used by p_utils
    "data_version": "3",

    "playing_time__gte": 45,
    "count_match__gte": 5,
}

data = client.get_physical(params=params)
df = pd.DataFrame(data)

before = set(df.columns)
p_utils.add_standard_metrics(df)     # mutates df (adds columns)
added = sorted(set(df.columns) - before)

print("Added columns:")
print(added)

print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "laliga_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)

Added columns:
['accel_count_full_all', 'accel_count_full_bip', 'accel_count_full_otip', 'accel_count_full_tip', 'accel_count_per_30_otip', 'accel_count_per_30_tip', 'accel_count_per_60_bip', 'accel_count_per_90', 'decel_count_full_all', 'decel_count_full_bip', 'decel_count_full_otip', 'decel_count_full_tip', 'decel_count_per_30_otip', 'decel_count_per_30_tip', 'decel_count_per_60_bip', 'decel_count_per_90', 'distance_per_sprint', 'distance_per_sprint_bip', 'distance_per_sprint_otip', 'distance_per_sprint_tip', 'hi_count_full_bip', 'hi_count_per_30_otip', 'hi_count_per_30_tip', 'hi_count_per_60_bip', 'hi_count_per_90', 'hi_distance_full_bip', 'hi_distance_per_30_otip', 'hi_distance_per_30_tip', 'hi_distance_per_60_bip', 'hi_distance_per_90', 'hi_meters_per_minute', 'hi_meters_per_minute_bip', 'hi_meters_per_minute_otip', 'hi_meters_per_minute_tip', 'highaccel_count_full_bip', 'highaccel_count_per_30_otip', 'highaccel_count_per_30_tip', 'highaccel_count_per_60_bip', 'highaccel_count_per

In [14]:
# LA LIGA 2

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

params = {
    "season": 129,
    "competition": 42,
    "group_by": "player,team,competition,season,group",

    # required so minutes_full_tip exists
    "possession": "all,tip,otip",

    # often required to get the v3 schema fields used by p_utils
    "data_version": "3",

    "playing_time__gte": 45,
    "count_match__gte": 5,
}

data = client.get_physical(params=params)
df = pd.DataFrame(data)

before = set(df.columns)
p_utils.add_standard_metrics(df)     # mutates df (adds columns)
added = sorted(set(df.columns) - before)

print("Added columns:")
print(added)

print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "laliga2_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)

Added columns:
['accel_count_full_all', 'accel_count_full_bip', 'accel_count_full_otip', 'accel_count_full_tip', 'accel_count_per_30_otip', 'accel_count_per_30_tip', 'accel_count_per_60_bip', 'accel_count_per_90', 'decel_count_full_all', 'decel_count_full_bip', 'decel_count_full_otip', 'decel_count_full_tip', 'decel_count_per_30_otip', 'decel_count_per_30_tip', 'decel_count_per_60_bip', 'decel_count_per_90', 'distance_per_sprint', 'distance_per_sprint_bip', 'distance_per_sprint_otip', 'distance_per_sprint_tip', 'hi_count_full_bip', 'hi_count_per_30_otip', 'hi_count_per_30_tip', 'hi_count_per_60_bip', 'hi_count_per_90', 'hi_distance_full_bip', 'hi_distance_per_30_otip', 'hi_distance_per_30_tip', 'hi_distance_per_60_bip', 'hi_distance_per_90', 'hi_meters_per_minute', 'hi_meters_per_minute_bip', 'hi_meters_per_minute_otip', 'hi_meters_per_minute_tip', 'highaccel_count_full_bip', 'highaccel_count_per_30_otip', 'highaccel_count_per_30_tip', 'highaccel_count_per_60_bip', 'highaccel_count_per

In [15]:
# SERIE A

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

params = {
    "season": 129,
    "competition": 5,
    "group_by": "player,team,competition,season,group",

    # required so minutes_full_tip exists
    "possession": "all,tip,otip",

    # often required to get the v3 schema fields used by p_utils
    "data_version": "3",

    "playing_time__gte": 45,
    "count_match__gte": 5,
}

data = client.get_physical(params=params)
df = pd.DataFrame(data)

before = set(df.columns)
p_utils.add_standard_metrics(df)     # mutates df (adds columns)
added = sorted(set(df.columns) - before)

print("Added columns:")
print(added)

print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "seriea_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)

Added columns:
['accel_count_full_all', 'accel_count_full_bip', 'accel_count_full_otip', 'accel_count_full_tip', 'accel_count_per_30_otip', 'accel_count_per_30_tip', 'accel_count_per_60_bip', 'accel_count_per_90', 'decel_count_full_all', 'decel_count_full_bip', 'decel_count_full_otip', 'decel_count_full_tip', 'decel_count_per_30_otip', 'decel_count_per_30_tip', 'decel_count_per_60_bip', 'decel_count_per_90', 'distance_per_sprint', 'distance_per_sprint_bip', 'distance_per_sprint_otip', 'distance_per_sprint_tip', 'hi_count_full_bip', 'hi_count_per_30_otip', 'hi_count_per_30_tip', 'hi_count_per_60_bip', 'hi_count_per_90', 'hi_distance_full_bip', 'hi_distance_per_30_otip', 'hi_distance_per_30_tip', 'hi_distance_per_60_bip', 'hi_distance_per_90', 'hi_meters_per_minute', 'hi_meters_per_minute_bip', 'hi_meters_per_minute_otip', 'hi_meters_per_minute_tip', 'highaccel_count_full_bip', 'highaccel_count_per_30_otip', 'highaccel_count_per_30_tip', 'highaccel_count_per_60_bip', 'highaccel_count_per

In [16]:
# TURQUÍA

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

params = {
    "season": 129,
    "competition": 29,
    "group_by": "player,team,competition,season,group",

    # required so minutes_full_tip exists
    "possession": "all,tip,otip",

    # often required to get the v3 schema fields used by p_utils
    "data_version": "3",

    "playing_time__gte": 45,
    "count_match__gte": 5,
}

data = client.get_physical(params=params)
df = pd.DataFrame(data)

before = set(df.columns)
p_utils.add_standard_metrics(df)     # mutates df (adds columns)
added = sorted(set(df.columns) - before)

print("Added columns:")
print(added)

print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "tur_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)

Added columns:
['accel_count_full_all', 'accel_count_full_bip', 'accel_count_full_otip', 'accel_count_full_tip', 'accel_count_per_30_otip', 'accel_count_per_30_tip', 'accel_count_per_60_bip', 'accel_count_per_90', 'decel_count_full_all', 'decel_count_full_bip', 'decel_count_full_otip', 'decel_count_full_tip', 'decel_count_per_30_otip', 'decel_count_per_30_tip', 'decel_count_per_60_bip', 'decel_count_per_90', 'distance_per_sprint', 'distance_per_sprint_bip', 'distance_per_sprint_otip', 'distance_per_sprint_tip', 'hi_count_full_bip', 'hi_count_per_30_otip', 'hi_count_per_30_tip', 'hi_count_per_60_bip', 'hi_count_per_90', 'hi_distance_full_bip', 'hi_distance_per_30_otip', 'hi_distance_per_30_tip', 'hi_distance_per_60_bip', 'hi_distance_per_90', 'hi_meters_per_minute', 'hi_meters_per_minute_bip', 'hi_meters_per_minute_otip', 'hi_meters_per_minute_tip', 'highaccel_count_full_bip', 'highaccel_count_per_30_otip', 'highaccel_count_per_30_tip', 'highaccel_count_per_60_bip', 'highaccel_count_per

In [17]:
# UEFA CHAMPIONS LEAGUE


from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

params = {
    "season": 129,
    "competition": 10,
    "group_by": "player,team,competition,season,group",

    # required so minutes_full_tip exists
    "possession": "all,tip,otip",

    # often required to get the v3 schema fields used by p_utils
    "data_version": "3",

    "playing_time__gte": 45,
    "count_match__gte": 5,
}

data = client.get_physical(params=params)
df = pd.DataFrame(data)

before = set(df.columns)
p_utils.add_standard_metrics(df)     # mutates df (adds columns)
added = sorted(set(df.columns) - before)

print("Added columns:")
print(added)

print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "ucl_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)

Added columns:
['accel_count_full_all', 'accel_count_full_bip', 'accel_count_full_otip', 'accel_count_full_tip', 'accel_count_per_30_otip', 'accel_count_per_30_tip', 'accel_count_per_60_bip', 'accel_count_per_90', 'decel_count_full_all', 'decel_count_full_bip', 'decel_count_full_otip', 'decel_count_full_tip', 'decel_count_per_30_otip', 'decel_count_per_30_tip', 'decel_count_per_60_bip', 'decel_count_per_90', 'distance_per_sprint', 'distance_per_sprint_bip', 'distance_per_sprint_otip', 'distance_per_sprint_tip', 'hi_count_full_bip', 'hi_count_per_30_otip', 'hi_count_per_30_tip', 'hi_count_per_60_bip', 'hi_count_per_90', 'hi_distance_full_bip', 'hi_distance_per_30_otip', 'hi_distance_per_30_tip', 'hi_distance_per_60_bip', 'hi_distance_per_90', 'hi_meters_per_minute', 'hi_meters_per_minute_bip', 'hi_meters_per_minute_otip', 'hi_meters_per_minute_tip', 'highaccel_count_full_bip', 'highaccel_count_per_30_otip', 'highaccel_count_per_30_tip', 'highaccel_count_per_60_bip', 'highaccel_count_per

In [18]:
# UEFA EUROPA LEAGUE


from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

params = {
    "season": 129,
    "competition": 9,
    "group_by": "player,team,competition,season,group",

    # required so minutes_full_tip exists
    "possession": "all,tip,otip",

    # often required to get the v3 schema fields used by p_utils
    "data_version": "3",

    "playing_time__gte": 45,
    "count_match__gte": 5,
}

data = client.get_physical(params=params)
df = pd.DataFrame(data)

before = set(df.columns)
p_utils.add_standard_metrics(df)     # mutates df (adds columns)
added = sorted(set(df.columns) - before)

print("Added columns:")
print(added)

print("Has minutes_full_tip?", "minutes_full_tip" in df.columns)

desktop_path = os.path.join(DATA_DIR, "physical")
os.makedirs(desktop_path, exist_ok=True)

file_path = os.path.join(desktop_path, "uel_physical_standardized.csv")
df.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Shape:", df.shape)

Added columns:
['accel_count_full_all', 'accel_count_full_bip', 'accel_count_full_otip', 'accel_count_full_tip', 'accel_count_per_30_otip', 'accel_count_per_30_tip', 'accel_count_per_60_bip', 'accel_count_per_90', 'decel_count_full_all', 'decel_count_full_bip', 'decel_count_full_otip', 'decel_count_full_tip', 'decel_count_per_30_otip', 'decel_count_per_30_tip', 'decel_count_per_60_bip', 'decel_count_per_90', 'distance_per_sprint', 'distance_per_sprint_bip', 'distance_per_sprint_otip', 'distance_per_sprint_tip', 'hi_count_full_bip', 'hi_count_per_30_otip', 'hi_count_per_30_tip', 'hi_count_per_60_bip', 'hi_count_per_90', 'hi_distance_full_bip', 'hi_distance_per_30_otip', 'hi_distance_per_30_tip', 'hi_distance_per_60_bip', 'hi_distance_per_90', 'hi_meters_per_minute', 'hi_meters_per_minute_bip', 'hi_meters_per_minute_otip', 'hi_meters_per_minute_tip', 'highaccel_count_full_bip', 'highaccel_count_per_30_otip', 'highaccel_count_per_30_tip', 'highaccel_count_per_60_bip', 'highaccel_count_per

In [ ]:
# PARAGUAY
#
# División Profesional, comp 140, calendar-year league -- pull 2025 (128) and 2026 (130).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 128=2025, 130=2026
for season_id in [128, 130]:
    params = {
        "season": season_id,
        "competition": 140,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 140: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "paraguay_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# URUGUAY
#
# 2025 (season 128): single comp 95. 2026 (season 130): split into Primera Division (95) + Primera Division Clausura (655, a new id that only exists for 2026).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_plan = [
    (128, [95]),
    (130, [95, 655]),
]

season_dfs = []

for season_id, comp_ids in season_plan:
    dfs = []
    for comp_id in comp_ids:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }
        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    if len(comp_ids) > 1:
        # season_id/competition_id (and their _name siblings) excluded too --
        # they're numeric, so without this they'd get summed across the two
        # comp_ids just like a real metric would, corrupting the season join
        # key downstream (see arg/col/ligamx cells for where this bug was
        # first caught). Rate/peak metrics (PSV99 etc.) use max, cumulative
        # counters use sum.
        meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                     "season_id", "season_name", "competition_id", "competition_name"]
        rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
        numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
        max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
        sum_cols = [c for c in numeric_cols if c not in max_cols]
        other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

        df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
            **{col: (col, "last") for col in other_meta},
            **{col: (col, "sum") for col in sum_cols},
            **{col: (col, "max") for col in max_cols}
        )
    else:
        df_season = df

    before = set(df_season.columns)
    p_utils.add_standard_metrics(df_season)
    added = sorted(set(df_season.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df_season.columns)

    season_dfs.append(df_season)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "uruguay_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# PERU
#
# 2025 (season 128): single comp 139. 2026 (season 130): split into Primera Division (139) + Primera Division Clausura (652, a new id that only exists for 2026).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_plan = [
    (128, [139]),
    (130, [139, 652]),
]

season_dfs = []

for season_id, comp_ids in season_plan:
    dfs = []
    for comp_id in comp_ids:
        params = {
            "season": season_id,
            "competition": comp_id,
            "group_by": "player,team,competition,season,group",
            "possession": "all,tip,otip",
            "data_version": "3",
            "playing_time__gte": 45,
            "count_match__gte": 5,
        }
        data = client.get_physical(params=params)
        df_comp = pd.DataFrame(data)
        dfs.append(df_comp)
        print(f"Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    if len(comp_ids) > 1:
        # season_id/competition_id (and their _name siblings) excluded too --
        # they're numeric, so without this they'd get summed across the two
        # comp_ids just like a real metric would, corrupting the season join
        # key downstream (see arg/col/ligamx cells for where this bug was
        # first caught). Rate/peak metrics (PSV99 etc.) use max, cumulative
        # counters use sum.
        meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                     "season_id", "season_name", "competition_id", "competition_name"]
        rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
        numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
        max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
        sum_cols = [c for c in numeric_cols if c not in max_cols]
        other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

        df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
            **{col: (col, "last") for col in other_meta},
            **{col: (col, "sum") for col in sum_cols},
            **{col: (col, "max") for col in max_cols}
        )
    else:
        df_season = df

    before = set(df_season.columns)
    p_utils.add_standard_metrics(df_season)
    added = sorted(set(df_season.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df_season.columns)

    season_dfs.append(df_season)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "peru_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# CHAMPIONSHIP (ENG)
#
# Cross-year league, comp 31 -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 31,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 31: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "championship_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# BUNDESLIGA
#
# Cross-year league, comp 6 -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 6,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 6: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "bundesliga_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# 2. BUNDESLIGA
#
# Cross-year league, comp 40 -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 40,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 40: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "bundesliga2_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# LIGUE 1
#
# Cross-year league, comp 3 -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 3,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 3: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "ligue1_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# EREDIVISIE
#
# Cross-year league, comp 17 -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 17,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 17: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "eredivisie_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# BÉLGICA
#
# Pro League, comp 16, cross-year league -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 16,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 16: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "belgica_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# PORTUGAL
#
# Primeira Liga, comp 25, cross-year league -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 25,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 25: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "portugal_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# ESCOCIA
#
# Premiership, comp 18, cross-year league -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 18,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 18: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "escocia_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)


In [ ]:
# RUSIA
#
# Premier League, comp 27, cross-year league -- pull 2025/2026 (129) and 2026/2027 (131).

from skillcorner.client import SkillcornerClient
from skillcornerviz.utils import skillcorner_physical_utils as p_utils
import pandas as pd
import os

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

season_dfs = []

# 129=2025/2026, 131=2026/2027
for season_id in [129, 131]:
    params = {
        "season": season_id,
        "competition": 27,
        "group_by": "player,team,competition,season,group",
        "possession": "all,tip,otip",
        "data_version": "3",
        "playing_time__gte": 45,
        "count_match__gte": 5,
    }
    data = client.get_physical(params=params)
    df = pd.DataFrame(data)
    print(f"Season {season_id}, Competition 27: {len(df)} rows")
    if df.empty:
        print(f"Season {season_id}: no rows returned, skipping.")
        continue

    before = set(df.columns)
    p_utils.add_standard_metrics(df)
    added = sorted(set(df.columns) - before)
    print(f"Season {season_id} -- Added columns:", added)
    print(f"Season {season_id} -- Has minutes_full_tip?", "minutes_full_tip" in df.columns)

    season_dfs.append(df)

if not season_dfs:
    print("No data returned for any season -- league may not be licensed/tracked for physical data. Skipping file write.")
else:
    df = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)

    file_path = os.path.join(desktop_path, "rusia_physical_standardized.csv")
    df.to_csv(file_path, index=False)

    print("Saved to:", file_path)
    print("Shape:", df.shape)
